# Pipeline Tag Prediction
## Tagging the Untagged Model Cards with the Baseline Model

**DATASCI 266: Natural Language Processing with Deep Learning**

UC Berkeley, School of Information

---

This notebook takes `model_cards_untagged_cleaned.parquet` (the cleaned set of model cards that never had a `pipeline_tag` on the Hub) and runs them through the already-trained TF-IDF + Logistic Regression baseline to predict a tag and a confidence score for each one.

Loads the saved model artifacts directly, no retraining:
- `tfidf_vectorizer.joblib`
- `label_encoder.joblib`
- `lr_baseline_model.joblib`

Steps:
1. Load the saved baseline model artifacts
2. Load the untagged dataset
3. Clean text the same way as training
4. Predict tags and confidence scores
5. Assemble and save the final dataset

## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import re
import warnings
import joblib

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 120)

print('Libraries loaded check')

Libraries loaded check


## 1. Load the Saved Baseline Model Artifacts

Loading the fitted vectorizer, label encoder, and Logistic Regression model straight from Drive. These were saved after training the baseline notebook, so this is the exact same model already evaluated there, not a retrained copy.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/266-pipeline-tag-prediction'

tf_idf_vec = joblib.load(f'{DRIVE_DIR}/tfidf_vectorizer.joblib')
label_encoder = joblib.load(f'{DRIVE_DIR}/label_encoder.joblib')
lr_model = joblib.load(f'{DRIVE_DIR}/lr_baseline_model.joblib')

print('Loaded tfidf_vectorizer.joblib, label_encoder.joblib, lr_baseline_model.joblib')
print(f'Classes: {list(label_encoder.classes_)}')

Mounted at /content/drive
Loaded tfidf_vectorizer.joblib, label_encoder.joblib, lr_baseline_model.joblib
Classes: ['automatic-speech-recognition', 'image-classification', 'image-text-to-text', 'robotics', 'sentence-similarity', 'text-classification', 'text-generation', 'text-to-image', 'token-classification', 'translation']


## 2. Load the Untagged Dataset

This is the dataset built in the previous cleaning notebook: `model_cards_untagged_cleaned.parquet`, columns `modelId`, `text`, `char_len`, `word_count`.

In [4]:
df_untagged = pd.read_parquet(f'{DRIVE_DIR}/model_cards_untagged_cleaned.parquet')
print(f'Loaded untagged dataset: {df_untagged.shape}')
df_untagged.head()

Loaded untagged dataset: (115620, 4)


,modelId,text,char_len,word_count
0,DreadPoor/Kitsch_Late_ALT-TEST-Q5_K_M-GGUF,# DreadPoor/Kitsch_Late_ALT-TEST-Q5_K_M-GGUF\n\n![image](https://cdn-uploads.huggingface.co/production/uploads/63214...,1820,182
1,namlevan888/blockassist-bc-lethal_durable_raven_1761941295,# Gensyn BlockAssist\n\nGensyn's BlockAssist is a distributed extension of the paper [AssistanceZero: Scalably Solvi...,169,17
2,ElenaSenger/career-path-representation-mpnet-decorte,# career-path-representation-mpnet-decorte\nThis is a fine-tuned version of [sentence-transformers/all-mpnet-base-v2...,842,67
3,priorcomputers/llama-3.2-1b-instruct-cn-dat-kr0.05-a1.0-creative,# llama-3.2-1b-instruct-cn-dat-kr0.05-a1.0-creative\n\nThis is a **CreativityNeuro (CN)** modified version of [meta-...,1436,139
4,mradermacher/ACC-Qwen3-30B-A3B-i1-GGUF,## About\n\n<!-- ### quantize_version: 2 -->\n<!-- ### output_tensor_quantised: 1 -->\n<!-- ### convert_type: hf -->...,5703,488


## 3. Clean Text the Same Way as Training

Applying the identical `clean_text` function used on the training data in the baseline notebook, so the untagged cards get vectorized on the same footing as what the model was trained on. This has to be run again here even though the model itself is preloaded, since cleaning happens before vectorization, not inside the saved vectorizer.

In [5]:
# Same clean_text function used in the baseline notebook
def clean_text(text: str) -> str:
    """Lowercase, remove punctuation/numbers, collapse whitespace."""
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_untagged['text_clean'] = df_untagged['text'].apply(clean_text)
df_untagged[['modelId', 'text_clean']].head()

,modelId,text_clean
0,DreadPoor/Kitsch_Late_ALT-TEST-Q5_K_M-GGUF,dreadpoor kitsch late alt test q k m gguf image https cdn uploads huggingface co production uploads f c da c dccde v...
1,namlevan888/blockassist-bc-lethal_durable_raven_1761941295,gensyn blockassist gensyn s blockassist is a distributed extension of the paper assistancezero scalably solving assi...
2,ElenaSenger/career-path-representation-mpnet-decorte,career path representation mpnet decorte this is a fine tuned version of sentence transformers all mpnet base v http...
3,priorcomputers/llama-3.2-1b-instruct-cn-dat-kr0.05-a1.0-creative,llama b instruct cn dat kr a creative this is a creativityneuro cn modified version of meta llama llama b instruct h...
4,mradermacher/ACC-Qwen3-30B-A3B-i1-GGUF,about quantize version output tensor quantised convert type hf vocab type tags nicoboss quants q k iq m q k s iq xxs...


## 4. Predict Tags and Confidence Scores

Transforming the untagged cards with the already-fitted vectorizer (`.transform`, not `.fit_transform`, since it was fit during training and must not be refit here), then using `predict_proba` to get a probability distribution over the 10 classes for each row. The predicted tag is the class with the highest probability, and the confidence percentage is that probability.

In [6]:
X_untagged = tf_idf_vec.transform(df_untagged['text_clean'])

pred_probs = lr_model.predict_proba(X_untagged)
pred_class_idx = np.argmax(pred_probs, axis=1)
pred_confidence = np.max(pred_probs, axis=1)

df_untagged['predicted_tag'] = label_encoder.inverse_transform(pred_class_idx)
df_untagged['confidence'] = pred_confidence
df_untagged['confidence_pct'] = (df_untagged['confidence'] * 100).round(2)

print('Predictions complete.')
df_untagged[['modelId', 'predicted_tag', 'confidence_pct']].head(10)

Predictions complete.


,modelId,predicted_tag,confidence_pct
0,DreadPoor/Kitsch_Late_ALT-TEST-Q5_K_M-GGUF,image-text-to-text,42.78
1,namlevan888/blockassist-bc-lethal_durable_raven_1761941295,text-generation,30.50
2,ElenaSenger/career-path-representation-mpnet-decorte,sentence-similarity,87.87
3,priorcomputers/llama-3.2-1b-instruct-cn-dat-kr0.05-a1.0-creative,text-generation,97.08
4,mradermacher/ACC-Qwen3-30B-A3B-i1-GGUF,text-generation,43.50
5,4everStudent/Qwen3-4B-GRPO-chess-puzzle,text-generation,89.23
6,javasop/orbital-cli,text-generation,26.54
7,little1d/C,robotics,30.98
8,munish0838/Qwen-2.5-1.5B-cenv-trl-grpo-v3,text-classification,27.03
9,phanerozoic/threshold-parity6,text-generation,27.54


A quick look at the confidence distribution and predicted tag counts, useful to sanity check whether the model is confident overall or hedging a lot on this unseen population of cards.

In [7]:
print(df_untagged['confidence_pct'].describe())
print()
print(df_untagged['predicted_tag'].value_counts())

count    115620.000000
mean         63.595837
std          24.901461
min          14.140000
25%          39.810000
50%          64.090000
75%          88.360000
max         100.000000
Name: confidence_pct, dtype: float64

predicted_tag
text-generation                 70027
image-text-to-text              12758
image-classification             8163
text-classification              7710
text-to-image                    5160
robotics                         4682
automatic-speech-recognition     3809
sentence-similarity              1198
token-classification             1097
translation                      1016
Name: count, dtype: int64


In [8]:
# Spot check a handful of low-confidence predictions, these are the ones most worth a manual look later
df_untagged.sort_values('confidence_pct').head(5)[['modelId', 'predicted_tag', 'confidence_pct', 'text']]

,modelId,predicted_tag,confidence_pct,text
103284,manak0/Detect-number-plates-1-0-winner,token-classification,14.14,# Detect-number-plates-1-0-winner\n\nPublished winning miner converted into a library element.\n\n- Source winner re...
76765,cplusx/MEMO_onnx_runtime,automatic-speech-recognition,14.47,# MEMO ONNX Runtime Models\n\nThis repository stores exported ONNX Runtime model files for MEMO.\n\n## Included Mode...
27211,uyen1109/eth-fraud-gnn-uyenuyen-v1,image-classification,14.65,# Ethereum Fraud GNN – Gradio Demo (v1)\n\nDemo tra cứu nhanh xác suất nghi vấn của node Ethereum (từ bảng điểm đã x...
54920,jasonmusespresso/glove-2024-wikigiga-100d,token-classification,14.81,"# GloVe 2024 WikiGigaword 100d\n\nThis is a mirror of the GloVe word vectors from Stanford NLP, hosted here for fast..."
82049,medicenjona1/jona,token-classification,14.87,"# 🗣️ medicenjona1 - Jona (Cloned Voice Model)\n\nEste es un modelo de clonación de voz de **medicenjona1 (Jona)**, u..."


## 5. Assemble and Save the Final Dataset

Final columns: `modelId`, the cleaned card text, and the predicted tag with confidence. Keeping both `predicted_tag` and `confidence_pct` rather than folding them into one string, so the confidence stays usable as a number for filtering or thresholding later.

Saved as both a parquet (consistent with the rest of the project's files) and a CSV, since a CSV was specifically requested.

In [9]:
final_df = df_untagged[['modelId', 'text_clean', 'predicted_tag', 'confidence_pct']].copy()
final_df = final_df.rename(columns={'text_clean': 'card_text_clean'})

print(f'Final dataset shape: {final_df.shape}')
final_df.head()

Final dataset shape: (115620, 4)


,modelId,card_text_clean,predicted_tag,confidence_pct
0,DreadPoor/Kitsch_Late_ALT-TEST-Q5_K_M-GGUF,dreadpoor kitsch late alt test q k m gguf image https cdn uploads huggingface co production uploads f c da c dccde v...,image-text-to-text,42.78
1,namlevan888/blockassist-bc-lethal_durable_raven_1761941295,gensyn blockassist gensyn s blockassist is a distributed extension of the paper assistancezero scalably solving assi...,text-generation,30.50
2,ElenaSenger/career-path-representation-mpnet-decorte,career path representation mpnet decorte this is a fine tuned version of sentence transformers all mpnet base v http...,sentence-similarity,87.87
3,priorcomputers/llama-3.2-1b-instruct-cn-dat-kr0.05-a1.0-creative,llama b instruct cn dat kr a creative this is a creativityneuro cn modified version of meta llama llama b instruct h...,text-generation,97.08
4,mradermacher/ACC-Qwen3-30B-A3B-i1-GGUF,about quantize version output tensor quantised convert type hf vocab type tags nicoboss quants q k iq m q k s iq xxs...,text-generation,43.50


In [10]:
final_df.to_parquet(f'{DRIVE_DIR}/untagged_model_cards_predicted.parquet', index=False)
final_df.to_csv(f'{DRIVE_DIR}/untagged_model_cards_predicted.csv', index=False)

print(f'Saved to {DRIVE_DIR}/untagged_model_cards_predicted.parquet')
print(f'Saved to {DRIVE_DIR}/untagged_model_cards_predicted.csv')

Saved to /content/drive/MyDrive/266-pipeline-tag-prediction/untagged_model_cards_predicted.parquet
Saved to /content/drive/MyDrive/266-pipeline-tag-prediction/untagged_model_cards_predicted.csv


### Notes for later use

- These are baseline-model predictions, not ground truth. Treat `confidence_pct` as a filter, not a guarantee, low-confidence rows are the ones most likely to be genuinely ambiguous or out of distribution relative to the training set.
- Since the baseline was trained only on the top 10 pipeline tags, every prediction here is forced into one of those 10 categories, even if a card actually belongs to a tag outside that set. Worth flagging as a limitation if this dataset gets used downstream.
- If `tfidf_vectorizer.joblib`, `label_encoder.joblib`, or `lr_baseline_model.joblib` are missing from Drive, this notebook will fail at Section 1. In that case they need to be regenerated from the baseline notebook and saved with `joblib.dump` before rerunning this.